# QUAL-005 - Entrenamiento final IA v5 cloud portable

Soporta Google Colab y Google Compute Engine. La v4 permanece intacta como referencia Colab validada. En VM los datasets se leen desde disco local (`/opt/pfi`); GCS es persistencia, no filesystem de entrenamiento. Los checkpoints se guardan atomico, el resume se sincroniza por `PFI_RUN_ID`, `PFI_PREFLIGHT_ONLY` evita entrenamiento, y este notebook no crea la VM.


## Diferencias controladas respecto de v4

- runtime portable;
- rutas separadas para modelos, resume, manifests y logs;
- rebasing axial de paths heredados;
- guardado atomico;
- sync de resume/final;
- preflight-only;
- outputs limpios.

No se cambian arquitectura, loss, optimizer, scheduler, metricas, splits, hiperparametros, dataset completo ni quality gate.


## 0. Bootstrap portable / repo


In [ ]:
from __future__ import annotations
import os, subprocess, sys
from pathlib import Path

PFI_REPO_URL = os.getenv("PFI_REPO_URL", "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git")

def _detect_runtime_bootstrap() -> str:
    if os.getenv("PFI_CLOUD_MODE") == "1": return "gce"
    try:
        import google.colab  # type: ignore
        return "colab"
    except Exception:
        return "local"

RUNTIME = _detect_runtime_bootstrap()
if RUNTIME == "gce":
    if not os.getenv("PFI_REPO_ROOT"):
        raise RuntimeError("PFI_REPO_ROOT es obligatorio en GCE")
    PFI_REPO_ROOT = Path(os.environ["PFI_REPO_ROOT"])
elif RUNTIME == "colab":
    PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
    if not PFI_REPO_ROOT.exists():
        print(f"Clonando AI Module en {PFI_REPO_ROOT} desde {PFI_REPO_URL}")
        subprocess.run(["git", "clone", PFI_REPO_URL, str(PFI_REPO_ROOT)], check=True)
else:
    PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", Path.cwd())).resolve()
    if not (PFI_REPO_ROOT / ".git").exists():
        probe = subprocess.run(["git", "rev-parse", "--show-toplevel"], text=True, capture_output=True)
        if probe.returncode == 0: PFI_REPO_ROOT = Path(probe.stdout.strip())

arch_path = PFI_REPO_ROOT / "ai_service" / "pfi_ai_service" / "model_architectures.py"
if not PFI_REPO_ROOT.exists() or not (PFI_REPO_ROOT / ".git").exists():
    raise FileNotFoundError(f"repo git inexistente: {PFI_REPO_ROOT}")
if not arch_path.exists():
    raise FileNotFoundError(f"No existe model_architectures.py en {arch_path}")
ai_service_path = str(PFI_REPO_ROOT / "ai_service")
if ai_service_path not in sys.path: sys.path.insert(0, ai_service_path)

def _git(args:list[str])->str:
    res=subprocess.run(["git", "-C", str(PFI_REPO_ROOT), *args], text=True, capture_output=True, check=False)
    return res.stdout.strip() if res.returncode==0 else "NO_OBTENIDO"
GIT_BRANCH=_git(["branch","--show-current"]); GIT_HEAD=_git(["rev-parse","HEAD"])
GIT_DIRTY=bool(_git(["status","--porcelain"]))
print({"runtime":RUNTIME,"repo_root":str(PFI_REPO_ROOT),"branch":GIT_BRANCH,"head":GIT_HEAD[:12],"dirty":GIT_DIRTY})


## 0.1 Dependencias y Google Drive seguros


In [ ]:
from pfi_ai_service.training.cloud_runtime import env_bool

INSTALL_NOTEBOOK_DEPS = env_bool("PFI_INSTALL_NOTEBOOK_DEPS", RUNTIME == "colab")
USE_GOOGLE_DRIVE = env_bool("PFI_USE_GOOGLE_DRIVE", RUNTIME == "colab")
if RUNTIME == "colab" and INSTALL_NOTEBOOK_DEPS:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "SimpleITK", "pydicom", "scikit-learn", "pandas", "pillow"], check=True)
else:
    print("Instalacion de dependencias omitida; prepare-training-vm.sh debe resolver faltantes en GCE/local.")

if RUNTIME == "colab" and USE_GOOGLE_DRIVE:
    from pathlib import Path
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        print("Google Drive ya montado; se reutiliza /content/drive/MyDrive")
    else:
        from google.colab import drive  # type: ignore
        mountpoint = Path("/content/drive")
        if mountpoint.exists() and any(mountpoint.iterdir()):
            raise RuntimeError("/content/drive existe y contiene archivos pero MyDrive no esta montado; no se limpia automaticamente")
        drive.mount(str(mountpoint))
elif RUNTIME == "gce":
    print("GCE: Google Drive deshabilitado")


## 1. Imports


In [ ]:
import json, os, random, sys, time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from typing import Any, Iterable, Literal

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
try:
    import SimpleITK as sitk
except Exception:
    sitk = None
try:
    import pydicom
except Exception:
    pydicom = None

from pfi_ai_service.training.cloud_runtime import (
    atomic_torch_save, atomic_write_dataframe_csv, atomic_write_json,
    build_sync_command, detect_runtime, env_bool, load_bash_env_contract,
    resolve_portable_axial_path, run_sync_command, validate_run_id,
    wait_for_minimum_file_age,
)
from pfi_ai_service.model_architectures import AxialUNet2D, SagittalUNet2D, build_checkpoint_model

def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


## 2. Configuracion portable


In [ ]:
@dataclass
class TrainConfig:
    repo_root: Path = PFI_REPO_ROOT
    runtime: str = RUNTIME
    cloud_mode: bool = field(default_factory=lambda: env_bool("PFI_CLOUD_MODE", RUNTIME == "gce"))
    preflight_only: bool = field(default_factory=lambda: env_bool("PFI_PREFLIGHT_ONLY", False if RUNTIME == "colab" else True))
    run_id: str = field(default_factory=lambda: validate_run_id(os.getenv("PFI_RUN_ID", "pfi-final-training-v1")))
    training_env_file: Path = field(default_factory=lambda: Path(os.getenv("PFI_TRAINING_ENV_FILE", "/opt/pfi/PFI_MVPTest_Enzo_AImodule/infra/gcp/training-vm.env")))
    target_size: tuple[int, int] = (256, 256)
    base_channels: int = int(os.getenv("PFI_BASE_CHANNELS", "16"))
    batch_size: int = int(os.getenv("PFI_BATCH_SIZE", "8"))
    max_epochs: int = int(os.getenv("PFI_MAX_EPOCHS", "80"))
    patience: int = int(os.getenv("PFI_EARLY_STOP_PATIENCE", "12"))
    lr: float = float(os.getenv("PFI_LR", "0.0008"))
    seed: int = int(os.getenv("PFI_SEED", "20260716"))
    num_workers: int = int(os.getenv("PFI_NUM_WORKERS", "0"))
    output_dir: Path = field(default_factory=lambda: Path(os.getenv("PFI_TRAIN_OUTPUT_DIR", "/content/drive/MyDrive/PFI_MVP/models/final_training")))
    models_dir: Path = field(default_factory=lambda: Path(os.getenv("PFI_LOCAL_MODELS_DIR", os.getenv("PFI_TRAIN_OUTPUT_DIR", "/content/drive/MyDrive/PFI_MVP/models/final_training"))))
    resume_dir: Path = field(default_factory=lambda: Path(os.getenv("PFI_LOCAL_RESUME_DIR", str(Path(os.getenv("PFI_TRAIN_OUTPUT_DIR", "/content/drive/MyDrive/PFI_MVP/models/final_training")) / "resume"))))
    manifests_dir: Path = field(default_factory=lambda: Path(os.getenv("PFI_LOCAL_MANIFESTS_DIR", os.getenv("PFI_TRAIN_OUTPUT_DIR", "/content/drive/MyDrive/PFI_MVP/models/final_training"))))
    logs_dir: Path = field(default_factory=lambda: Path(os.getenv("PFI_LOCAL_LOGS_DIR", os.getenv("PFI_TRAIN_OUTPUT_DIR", "/content/drive/MyDrive/PFI_MVP/models/final_training"))))
    sync_dry_run: bool = field(default_factory=lambda: env_bool("PFI_SYNC_DRY_RUN", True))
    sync_resume: bool = field(default_factory=lambda: env_bool("PFI_SYNC_RESUME", False))
    sync_final_artifacts: bool = field(default_factory=lambda: env_bool("PFI_SYNC_FINAL_ARTIFACTS", False))
    sync_every_n_epochs: int = int(os.getenv("PFI_SYNC_EVERY_N_EPOCHS", "1"))
    sync_failure_is_fatal: bool = field(default_factory=lambda: env_bool("PFI_SYNC_FAILURE_IS_FATAL", True))
    sync_min_file_age_seconds: int = int(os.getenv("PFI_SYNC_MIN_FILE_AGE_SECONDS", "10"))
    use_google_drive: bool = USE_GOOGLE_DRIVE
    install_notebook_deps: bool = INSTALL_NOTEBOOK_DEPS
    spider_images_dir: Path = Path(os.getenv("SPIDER_IMAGES_DIR", "/content/drive/MyDrive/PFI_MVP/data/SPIDER/images"))
    spider_masks_dir: Path = Path(os.getenv("SPIDER_MASKS_DIR", "/content/drive/MyDrive/PFI_MVP/data/SPIDER/masks"))
    spider_label_map_json: Path = Path(os.getenv("SPIDER_LABEL_GROUP_MAPPING_JSON", "/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado/E5_multiclass_label_mapping.json"))
    spider_checkpoint_for_label_map: Path = Path(os.getenv("SPIDER_CHECKPOINT_FOR_LABEL_MAP", "/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt"))
    spider_sagittal_axis: int = int(os.getenv("SPIDER_SAGITTAL_AXIS", "2"))
    axial_split_csv: Path = Path(os.getenv("AXIAL_E9_CURATED_SPLIT_CSV", "/content/drive/MyDrive/PFI_MVP/results/E9_alkafri_axial_t2_final_labels_baseline/E9_t2_final_labels_curated_split.csv"))
    axial_images_dir: Path = Path(os.getenv("AXIAL_IMAGES_DIR", "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI"))
    axial_masks_dir: Path = Path(os.getenv("AXIAL_MASKS_DIR", "/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI"))
    axial_image_col: str = os.getenv("AXIAL_IMAGE_COL", "image_file_path")
    axial_mask_col: str = os.getenv("AXIAL_MASK_COL", "final_label_file_path")
    axial_patient_col: str = os.getenv("AXIAL_PATIENT_COL", "case_id_norm")
    axial_split_col: str = os.getenv("AXIAL_SPLIT_COL", "split")
    require_gpu: bool = field(default_factory=lambda: env_bool("REQUIRE_GPU", True))
    axial_raw_pairing_mode: bool = field(default_factory=lambda: env_bool("AXIAL_RAW_PAIRING_MODE", False))
    smoke_run: bool = field(default_factory=lambda: env_bool("PFI_SMOKE_RUN", False))
    weight_estimate_max_records: int = int(os.getenv("PFI_WEIGHT_MAX_RECORDS", "128"))
    train_log_every_n_batches: int = int(os.getenv("PFI_LOG_EVERY_N_BATCHES", "25"))
    resume_training: bool = field(default_factory=lambda: env_bool("PFI_RESUME_TRAINING", True))

    def validate(self) -> None:
        validate_run_id(self.run_id)
        if self.sync_every_n_epochs < 1: raise ValueError("PFI_SYNC_EVERY_N_EPOCHS debe ser >= 1")
        if self.sync_min_file_age_seconds < 0: raise ValueError("PFI_SYNC_MIN_FILE_AGE_SECONDS debe ser >= 0")
        if self.cloud_mode:
            vm_root = Path(os.environ.get("PFI_VM_ROOT", "/opt/pfi"))
            for name in ["output_dir","models_dir","resume_dir","manifests_dir","logs_dir","training_env_file"]:
                p = getattr(self, name)
                if not p.is_absolute() or str(p) == "/": raise ValueError(f"{name} debe ser absoluto y no raiz: {p}")
            for name in ["output_dir","models_dir","resume_dir","manifests_dir","logs_dir"]:
                p = getattr(self, name)
                if not str(p).startswith(str(vm_root)): raise ValueError(f"{name} fuera de PFI_VM_ROOT: {p}")
                if not self.preflight_only and (not p.is_dir() or not os.access(p, os.W_OK)): raise FileNotFoundError(f"{name} no existe/escribible: {p}")
            if len({self.models_dir,self.resume_dir,self.manifests_dir,self.logs_dir}) != 4: raise ValueError("models/resume/manifests/logs deben ser distintos")
            if not self.training_env_file.exists(): raise FileNotFoundError(f"PFI_TRAINING_ENV_FILE inexistente: {self.training_env_file}")
            sync_script=self.repo_root/"infra"/"gcp"/"sync-training-artifacts.sh"
            if not sync_script.exists(): raise FileNotFoundError(f"sync script inexistente: {sync_script}")
        else:
            for p in [self.output_dir,self.models_dir,self.resume_dir,self.manifests_dir,self.logs_dir]: p.mkdir(parents=True, exist_ok=True)

CFG = TrainConfig(); CFG.validate()
if CFG.smoke_run:
    CFG.max_epochs = min(CFG.max_epochs, 1); CFG.patience = min(CFG.patience, 1)
print(CFG)

def set_seed(seed:int)->None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
set_seed(CFG.seed)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

def require_cuda_for_training() -> None:
    if CFG.require_gpu and DEVICE.type != "cuda":
        raise RuntimeError("REQUIRE_GPU=1 pero CUDA no esta disponible. Frenando entrenamiento final para evitar corrida CPU accidental.")


## 3. Labels, metricas y helpers preservados


In [ ]:
SAGITTAL_CLASSES=["background","vertebra_group","canal","disc_group"]
AXIAL_CLASSES=["background_250","raw_0","raw_50","raw_100","raw_150","raw_200"]
AXIAL_RAW_LABEL_MAP={250:0,0:1,50:2,100:3,150:4,200:5}

def _int_value(value, field:str)->int:
    try:
        return int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Mapping invalido: {field}={value!r} no es entero") from exc

def normalize_label_mapping(raw_mapping:dict, num_classes:int=4)->dict[int,int]:
    """Soporta raw_label -> class_id y class_id -> [raw_labels]."""
    out:dict[int,int]={}
    for raw_key, raw_value in raw_mapping.items():
        if isinstance(raw_value, (list, tuple, set)):
            class_id=_int_value(raw_key,"class_id")
            if class_id<0 or class_id>=num_classes:
                raise ValueError(f"class_id fuera de rango: {class_id}")
            for raw_label in raw_value:
                out[_int_value(raw_label,"raw_label")]=class_id
        else:
            raw_label=_int_value(raw_key,"raw_label")
            class_id=_int_value(raw_value,"class_id")
            if class_id<0 or class_id>=num_classes:
                raise ValueError(f"class_id fuera de rango: {class_id}")
            out[raw_label]=class_id
    out.setdefault(0,0)
    return dict(sorted(out.items()))

def load_json_mapping(path:Path, checkpoint_fallback:Path|None=None, num_classes:int=4)->dict[int,int]:
    if path.exists():
        data=json.loads(path.read_text())
        return normalize_label_mapping(data, num_classes=num_classes)
    if checkpoint_fallback is not None and checkpoint_fallback.exists():
        ckpt=torch.load(checkpoint_fallback,map_location="cpu",weights_only=False)
        mapping=ckpt.get("label_group_mapping") if isinstance(ckpt,dict) else None
        if mapping:
            return normalize_label_mapping(mapping, num_classes=num_classes)
    raise FileNotFoundError(f"Falta label_group_mapping real: {path}; fallback={checkpoint_fallback}")

def canonicalize_spider_array(arr:np.ndarray)->np.ndarray:
    arr=np.asarray(arr)
    if arr.ndim==3 and arr.shape[0]<=64 and arr.shape[-1]>64:
        return np.moveaxis(arr,0,-1)
    return arr

def apply_label_map(mask:np.ndarray, label_map:dict[int,int]|None, num_classes:int, case_id:str)->np.ndarray:
    mask=np.asarray(mask)
    if label_map is None:
        labels=sorted(int(v) for v in np.unique(mask)); bad=[v for v in labels if v<0 or v>=num_classes]
        if bad: raise ValueError(f"{case_id}: labels fuera de rango {bad}; proveer mapping")
        return mask.astype(np.int64)
    labels=sorted(int(v) for v in np.unique(mask)); missing=[v for v in labels if v not in label_map]
    if missing: raise ValueError(f"{case_id}: labels sin mapping {missing[:20]}")
    out=np.zeros(mask.shape,dtype=np.int64)
    for raw,cls in label_map.items():
        if cls<0 or cls>=num_classes: raise ValueError(f"class_id fuera de rango: {cls}")
        out[mask==raw]=cls
    return out

def read_array(path:Path)->np.ndarray:
    s=path.suffix.lower()
    if s==".npy": return np.load(path)
    if s in {".png",".jpg",".jpeg",".bmp",".tif",".tiff"}: return np.asarray(Image.open(path).convert("L"))
    if s in {".mha",".mhd"}:
        if sitk is None: raise ImportError("SimpleITK requerido")
        return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    if s in {".dcm",".ima"}:
        if pydicom is None: raise ImportError("pydicom requerido para DICOM/IMA")
        ds=pydicom.dcmread(str(path),force=True)
        return np.asarray(ds.pixel_array)
    raise ValueError(f"Formato no soportado: {path}")

def robust_normalize(x:np.ndarray)->np.ndarray:
    x=np.asarray(x,dtype=np.float32); finite=np.isfinite(x)
    if not finite.any(): return np.zeros_like(x,dtype=np.float32)
    lo,hi=np.percentile(x[finite],[1,99])
    if hi<=lo: return np.zeros_like(x,dtype=np.float32)
    return ((np.clip(x,lo,hi)-lo)/(hi-lo+1e-8)).astype(np.float32)

def resize_pair(image:np.ndarray, mask:np.ndarray, size:tuple[int,int])->tuple[np.ndarray,np.ndarray]:
    img=(robust_normalize(image)*255).clip(0,255).astype(np.uint8)
    img=Image.fromarray(img).resize((size[1],size[0]),Image.Resampling.BILINEAR)
    msk=Image.fromarray(mask.astype(np.int32),mode="I").resize((size[1],size[0]),Image.Resampling.NEAREST)
    return np.asarray(img,dtype=np.float32)/255.0, np.asarray(msk,dtype=np.int64)

def augment(image:np.ndarray, mask:np.ndarray)->tuple[np.ndarray,np.ndarray]:
    if random.random()<0.5: image,mask=np.fliplr(image).copy(),np.fliplr(mask).copy()
    if random.random()<0.2: image,mask=np.flipud(image).copy(),np.flipud(mask).copy()
    if random.random()<0.4:
        angle=random.uniform(-8,8)
        image=np.asarray(Image.fromarray((image*255).astype(np.uint8)).rotate(angle,Image.Resampling.BILINEAR),dtype=np.float32)/255.0
        mask=np.asarray(Image.fromarray(mask.astype(np.int32),mode="I").rotate(angle,Image.Resampling.NEAREST),dtype=np.int64)
    if random.random()<0.5:
        pil=Image.fromarray((image*255).clip(0,255).astype(np.uint8))
        image=np.asarray(ImageEnhance.Contrast(pil).enhance(random.uniform(0.85,1.15)),dtype=np.float32)/255.0
    return image,mask

def patient_id_from_stem(stem:str)->str:
    return stem.replace("-","_").split("_")[0] or stem

def split_patients(ids:Iterable[str], seed:int)->dict[str,str]:
    ids=sorted(set(ids)); train,tmp=train_test_split(ids,test_size=0.30,random_state=seed)
    val,test=train_test_split(tmp,test_size=0.50,random_state=seed)
    m={x:"train" for x in train}; m.update({x:"val" for x in val}); m.update({x:"test" for x in test}); return m


## 4. Datasets 2D, split por paciente y rebasing axial


In [ ]:
@dataclass(frozen=True)
class SliceRecord:
    image_path:str; mask_path:str; patient_id:str; split:Literal["train","val","test"]; plane:Literal["sagittal","axial"]; slice_index:int|None=None; slice_axis:int|None=None

class Segmentation2DDataset(Dataset):
    def __init__(self, records:list[SliceRecord], num_classes:int, label_map:dict[int,int]|None, augment_on:bool):
        self.records=records; self.num_classes=num_classes; self.label_map=label_map; self.augment_on=augment_on
    def __len__(self): return len(self.records)
    def __getitem__(self, idx:int):
        r=self.records[idx]; image=read_array(Path(r.image_path)); mask=read_array(Path(r.mask_path))
        if r.plane=="sagittal":
            image=canonicalize_spider_array(image)
            mask=canonicalize_spider_array(mask)
        if image.ndim==3:
            axis=r.slice_axis if r.slice_axis is not None else int(np.argmin(image.shape))
            si=r.slice_index if r.slice_index is not None else image.shape[axis]//2
            image=np.take(image,si,axis=axis); mask=np.take(mask,si,axis=axis)
        mask=apply_label_map(mask,self.label_map,self.num_classes,r.patient_id)
        image,mask=resize_pair(image,mask,CFG.target_size)
        if self.augment_on: image,mask=augment(image,mask)
        return torch.from_numpy(image[None]).float(), torch.from_numpy(mask).long()

def pair_by_stem(images_dir:Path,masks_dir:Path)->list[tuple[Path,Path,str]]:
    allowed={".mha",".mhd",".npy",".png",".tif",".tiff",".dcm",".ima"}
    images=[p for p in images_dir.rglob("*") if p.is_file() and p.suffix.lower() in allowed]
    masks=[p for p in masks_dir.rglob("*") if p.is_file() and p.suffix.lower() in allowed]
    mask_by={p.stem.replace("_mask","").replace("_label",""):p for p in masks}
    pairs=[]
    for img in sorted(images):
        key=img.stem.replace("_image","")
        if key in mask_by: pairs.append((img,mask_by[key],patient_id_from_stem(key)))
    if not pairs: raise FileNotFoundError(f"Sin pares en {images_dir} / {masks_dir}")
    return pairs

def foreground_indices(mask:np.ndarray, axis:int, max_slices:int=16, bg_ratio:float=0.20)->list[int]:
    if mask.ndim==2: return [0]
    count=mask.shape[axis]; fg=[i for i in range(count) if np.take(mask,i,axis=axis).max()>0]
    if len(fg)>max_slices: fg=[fg[i] for i in np.linspace(0,len(fg)-1,max_slices,dtype=int)]
    bg_pool=[i for i in range(count) if i not in set(fg)]
    n_bg=min(len(bg_pool),max(1,int(len(fg)*bg_ratio))) if fg else min(len(bg_pool),4)
    bg=random.sample(bg_pool,n_bg) if n_bg else []
    return sorted(set(fg+bg))

def build_spider_records()->tuple[list[SliceRecord],dict[int,int]]:
    label_map=load_json_mapping(CFG.spider_label_map_json, CFG.spider_checkpoint_for_label_map, num_classes=4)
    pairs=pair_by_stem(CFG.spider_images_dir,CFG.spider_masks_dir)
    p_split=split_patients([pid for _,_,pid in pairs],CFG.seed)
    records=[]
    for img,msk,pid in pairs:
        raw=canonicalize_spider_array(read_array(msk)); grouped=apply_label_map(raw,label_map,4,pid)
        axis=0 if grouped.ndim==2 else CFG.spider_sagittal_axis
        for si in foreground_indices(grouped,axis): records.append(SliceRecord(str(img),str(msk),pid,p_split[pid],"sagittal",si,axis))
    return records,label_map

@dataclass
class AxialPathStats:
    total:int=0; rebased:int=0; direct:int=0; missing:int=0
    rebased_examples:list[tuple[str,str]]=field(default_factory=list)
    missing_examples:list[str]=field(default_factory=list)

def resolve_axial_csv_path(value:Any, *, primary_root:Path, additional_roots:list[Path])->tuple[Path,bool]:
    return resolve_portable_axial_path(value, primary_root=primary_root, additional_roots=additional_roots)

def infer_axial_label_map(records:list[SliceRecord], max_masks:int=16)->dict[int,int]|None:
    labels:set[int]=set()
    for r in records[:max_masks]:
        arr=read_array(Path(r.mask_path))
        labels.update(int(v) for v in np.unique(arr))
        if any(v<0 or v>=len(AXIAL_CLASSES) for v in labels):
            break
    if labels and all(0<=v<len(AXIAL_CLASSES) for v in labels):
        print(f"Axial labels 0..{len(AXIAL_CLASSES)-1} detectados; no se aplica mapping. labels={sorted(labels)}")
        return None
    raw_keys=set(AXIAL_RAW_LABEL_MAP)
    if labels and labels.issubset(raw_keys):
        print(f"Axial labels crudos detectados; aplicando AXIAL_RAW_LABEL_MAP. labels={sorted(labels)}")
        return AXIAL_RAW_LABEL_MAP
    raise ValueError(f"Labels axiales no reconocidos {sorted(labels)}; esperado 0..5 o raw {sorted(raw_keys)}")

def build_axial_records()->tuple[list[SliceRecord],dict[int,int]|None]:
    if not CFG.axial_split_csv.exists(): raise FileNotFoundError(f"Falta split axial E9: {CFG.axial_split_csv}")
    df=pd.read_csv(CFG.axial_split_csv)
    for col in [CFG.axial_image_col,CFG.axial_mask_col]:
        if col not in df.columns: raise KeyError(f"Falta columna {col}; disponibles {list(df.columns)}")
    if CFG.axial_patient_col not in df.columns: df[CFG.axial_patient_col]=df[CFG.axial_image_col].map(lambda x: patient_id_from_stem(Path(str(x)).stem))
    if CFG.axial_split_col not in df.columns:
        p_split=split_patients(df[CFG.axial_patient_col].astype(str),CFG.seed); df[CFG.axial_split_col]=df[CFG.axial_patient_col].astype(str).map(p_split)
    df[CFG.axial_split_col]=df[CFG.axial_split_col].astype(str).str.lower()
    split_counts=df[CFG.axial_split_col].value_counts().to_dict()
    for required_split in ["train","val","test"]:
        if split_counts.get(required_split,0)==0:
            raise ValueError(f"Split axial vacio: {required_split}; counts={split_counts}")
    additional=[CFG.axial_split_csv.parent]
    records=[]; stats=AxialPathStats()
    for row in df.itertuples(index=False):
        raw_img=getattr(row,CFG.axial_image_col); raw_msk=getattr(row,CFG.axial_mask_col)
        img,img_rebased=resolve_axial_csv_path(raw_img, primary_root=CFG.axial_images_dir, additional_roots=additional)
        msk,msk_rebased=resolve_axial_csv_path(raw_msk, primary_root=CFG.axial_masks_dir, additional_roots=additional)
        stats.total += 2; stats.rebased += int(img_rebased)+int(msk_rebased); stats.direct += int(not img_rebased)+int(not msk_rebased)
        if img_rebased and len(stats.rebased_examples)<10: stats.rebased_examples.append((str(raw_img),str(img)))
        if msk_rebased and len(stats.rebased_examples)<10: stats.rebased_examples.append((str(raw_msk),str(msk)))
        missing=[]
        if not img.exists(): missing.append(str(img))
        if not msk.exists(): missing.append(str(msk))
        if missing:
            stats.missing += len(missing); stats.missing_examples.extend(missing[: max(0,10-len(stats.missing_examples))])
        records.append(SliceRecord(str(img),str(msk),str(getattr(row,CFG.axial_patient_col)),str(getattr(row,CFG.axial_split_col)),"axial"))
    print(f"AXIAL paths total={stats.total} rebased={stats.rebased} direct={stats.direct} missing={stats.missing}")
    if stats.rebased_examples: print("AXIAL rebased examples", stats.rebased_examples[:10])
    if stats.missing: raise FileNotFoundError(f"Paths axiales faltantes: {stats.missing_examples[:10]}")
    axial_label_map=AXIAL_RAW_LABEL_MAP if CFG.axial_raw_pairing_mode else infer_axial_label_map(records)
    return records, axial_label_map

def freeze_records(records:list[SliceRecord], path:Path)->None:
    atomic_write_dataframe_csv(pd.DataFrame([asdict(r) for r in records]), path, index=False); print(path)

def summarize_records(records:list[SliceRecord], title:str):
    df=pd.DataFrame([asdict(r) for r in records]); print(title); print(df.groupby("split").agg(n_slices=("image_path","count"),n_patients=("patient_id","nunique"))); return df


## 5. Preflight obligatorio y bloqueos de seguridad


In [ ]:
def validate_pre_training_security() -> None:
    if CFG.preflight_only:
        print("PREFLIGHT ONLY: entrenamiento desactivado.")
        return
    if CFG.cloud_mode:
        require_cuda_for_training()
        if not CFG.sync_resume: raise RuntimeError("PFI_SYNC_RESUME=1 requerido en cloud training")
        if CFG.sync_dry_run:
            raise RuntimeError("El entorno continua en dry-run. Validar primero los scripts y establecer PFI_SYNC_DRY_RUN=0 explicitamente para la corrida real.")
        keys=["PFI_RUN_ID","PFI_REPO_ROOT","PFI_LOCAL_RESUME_DIR","PFI_LOCAL_MODELS_DIR","PFI_LOCAL_MANIFESTS_DIR","PFI_LOCAL_LOGS_DIR","PFI_SYNC_RESUME","PFI_SYNC_FINAL_ARTIFACTS","PFI_SYNC_MIN_FILE_AGE_SECONDS"]
        contract=load_bash_env_contract(CFG.training_env_file, keys)
        expected={"PFI_RUN_ID":CFG.run_id,"PFI_REPO_ROOT":str(CFG.repo_root),"PFI_LOCAL_RESUME_DIR":str(CFG.resume_dir),"PFI_LOCAL_MODELS_DIR":str(CFG.models_dir),"PFI_LOCAL_MANIFESTS_DIR":str(CFG.manifests_dir),"PFI_LOCAL_LOGS_DIR":str(CFG.logs_dir)}
        mismatch={k:(contract.get(k),v) for k,v in expected.items() if contract.get(k)!=v}
        if mismatch: raise RuntimeError(f"Env contract mismatch: {mismatch}")

def preflight() -> None:
    print("== Repo ==", CFG.repo_root, GIT_BRANCH, GIT_HEAD[:12], "dirty", GIT_DIRTY)
    if not (CFG.repo_root / "ai_service" / "pfi_ai_service" / "model_architectures.py").exists(): raise FileNotFoundError("model_architectures.py no existe")
    print("== GPU ==", torch.cuda.is_available())
    if CFG.require_gpu and not torch.cuda.is_available(): print("WARNING: REQUIRE_GPU=1; entrenamiento final frenara sin CUDA.")
    print("== SPIDER ==")
    if not CFG.spider_images_dir.exists(): raise FileNotFoundError(f"SPIDER_IMAGES_DIR no existe: {CFG.spider_images_dir}")
    if not CFG.spider_masks_dir.exists(): raise FileNotFoundError(f"SPIDER_MASKS_DIR no existe: {CFG.spider_masks_dir}")
    mapping = load_json_mapping(CFG.spider_label_map_json, CFG.spider_checkpoint_for_label_map, num_classes=4)
    print("SPIDER mapping labels:", sorted(mapping.items())[:10], "... total", len(mapping))
    print("== Axial E9 ==")
    if not CFG.axial_split_csv.exists(): raise FileNotFoundError(f"CSV axial no existe: {CFG.axial_split_csv}")
    df = pd.read_csv(CFG.axial_split_csv)
    required_cols = {CFG.axial_image_col, CFG.axial_mask_col, CFG.axial_patient_col, CFG.axial_split_col}
    missing = required_cols - set(df.columns)
    if missing: raise KeyError(f"Columnas axiales faltantes: {missing}; columnas reales: {list(df.columns)}")
    counts = df[CFG.axial_split_col].astype(str).str.lower().value_counts().to_dict()
    for split_name in ["train", "val", "test"]:
        if counts.get(split_name, 0) == 0: raise ValueError(f"Split axial vacio: {split_name}; counts={counts}")
    rebased=0; missing_paths=[]
    for row in df.itertuples(index=False):
        img,ri=resolve_axial_csv_path(getattr(row,CFG.axial_image_col),primary_root=CFG.axial_images_dir,additional_roots=[CFG.axial_split_csv.parent])
        msk,rm=resolve_axial_csv_path(getattr(row,CFG.axial_mask_col),primary_root=CFG.axial_masks_dir,additional_roots=[CFG.axial_split_csv.parent])
        rebased += int(ri)+int(rm)
        if not img.exists(): missing_paths.append(str(img))
        if not msk.exists(): missing_paths.append(str(msk))
        if len(missing_paths)>10: break
    if missing_paths: raise FileNotFoundError(f"Paths axiales faltantes: {missing_paths[:10]}")
    print("Axial splits:", counts, "rebased", rebased)
    validate_pre_training_security()

preflight()
if CFG.preflight_only:
    raise SystemExit("PREFLIGHT ONLY: entrenamiento desactivado.")


## 6. Loss, pesos por clase, entrenamiento y checkpoints atomicos


In [ ]:
def dice_loss(logits:torch.Tensor,target:torch.Tensor,num_classes:int,eps:float=1e-6)->torch.Tensor:
    probs=torch.softmax(logits,dim=1); oh=F.one_hot(target,num_classes).permute(0,3,1,2).float()
    inter=(probs*oh).sum((0,2,3)); den=(probs+oh).sum((0,2,3)); dice=(2*inter+eps)/(den+eps)
    return 1-dice[1:].mean()

def estimate_weights(records:list[SliceRecord],num_classes:int,label_map:dict[int,int]|None,boost:dict[int,float]|None=None,max_records:int|None=None)->torch.Tensor:
    max_records = CFG.weight_estimate_max_records if max_records is None else max_records
    counts=np.ones(num_classes,dtype=np.float64); sample=records if len(records)<=max_records else random.sample(records,max_records)
    print(f"Estimando class_weights con {len(sample)}/{len(records)} muestras; PFI_WEIGHT_MAX_RECORDS={max_records}; num_workers={CFG.num_workers}")
    ds=Segmentation2DDataset(sample,num_classes,label_map,False)
    for i,(_,mask) in enumerate(ds, start=1):
        vals,freq=torch.unique(mask,return_counts=True)
        for v,f in zip(vals.tolist(),freq.tolist()): counts[int(v)]+=int(f)
        if i == 1 or i % 32 == 0 or i == len(sample):
            print(f"  class_weights sample {i}/{len(sample)}")
    w=(counts.sum()/counts); w=w/w.mean()
    if boost:
        for cid,factor in boost.items(): w[cid]*=factor
    print("class_weights", np.round(w, 3).tolist())
    return torch.tensor(np.clip(w,0.25,8.0),dtype=torch.float32)

def build_model(plane:str,num_classes:int,base_channels:int)->nn.Module:
    return SagittalUNet2D(num_classes=num_classes,base_channels=base_channels) if plane=="sagittal" else AxialUNet2D(num_classes=num_classes,base_channels=base_channels)

def evaluate(model:nn.Module,loader:DataLoader,num_classes:int)->dict[str,Any]:
    model.eval(); acc={c:{"intersection":0,"pred":0,"gt":0,"union":0,"gt_present_cases":0,"pred_present_cases":0} for c in range(num_classes)}
    with torch.inference_mode():
        for x,y in loader:
            pred=torch.argmax(torch.softmax(model(x.to(DEVICE)),dim=1),dim=1).cpu().numpy(); gt=y.numpy()
            for b in range(pred.shape[0]):
                for c in range(num_classes):
                    p=pred[b]==c; g=gt[b]==c
                    acc[c]["intersection"]+=int(np.logical_and(p,g).sum()); acc[c]["pred"]+=int(p.sum()); acc[c]["gt"]+=int(g.sum()); acc[c]["union"]+=int(np.logical_or(p,g).sum())
                    acc[c]["gt_present_cases"]+=int(g.any()); acc[c]["pred_present_cases"]+=int(p.any())
    per={}; dice_fg=[]; iou_fg=[]; dice_ex_raw0=[]
    for c,s in acc.items():
        dice=None if s["pred"]+s["gt"]==0 else 2*s["intersection"]/(s["pred"]+s["gt"])
        iou=None if s["union"]==0 else s["intersection"]/s["union"]
        per[str(c)]={**s,"dice":dice,"iou":iou}
        if c!=0 and dice is not None: dice_fg.append(dice)
        if c!=0 and iou is not None: iou_fg.append(iou)
        if c not in {0,1} and dice is not None: dice_ex_raw0.append(dice)
    return {"per_class":per,"dice_macro_no_bg":float(np.mean(dice_fg)) if dice_fg else None,"iou_macro_no_bg":float(np.mean(iou_fg)) if iou_fg else None,"dice_macro_excluding_raw0":float(np.mean(dice_ex_raw0)) if dice_ex_raw0 else None}

def loaders(records:list[SliceRecord],num_classes:int,label_map:dict[int,int]|None):
    split={s:[r for r in records if r.split==s] for s in ["train","val","test"]}
    for s,rs in split.items():
        if not rs: raise ValueError(f"Split vacio {s}")
    return tuple(DataLoader(Segmentation2DDataset(split[s],num_classes,label_map,s=="train"),batch_size=CFG.batch_size,shuffle=(s=="train"),num_workers=CFG.num_workers,pin_memory=torch.cuda.is_available() and CFG.num_workers>0) for s in ["train","val","test"])

def _cpu_state_dict(model:nn.Module)->dict[str,torch.Tensor]:
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def _resume_paths(output_name:str)->tuple[Path,Path]:
    stem=Path(output_name).stem
    return CFG.resume_dir/f"{stem}.last_checkpoint.pt", CFG.resume_dir/f"{stem}.best_checkpoint.pt"

def _checkpoint_common(*, epoch:int, plane:str, model_key:str, output_name:str, model_state:dict[str,torch.Tensor], opt, sch, best:float, best_state, bad:int, history:list[dict[str,Any]], weights:torch.Tensor, num_classes:int, monitor_metric:str, resume_kind:str)->dict[str,Any]:
    return {"checkpoint_schema_version":2,"run_id":CFG.run_id,"saved_at_utc":utc_now(),"git_commit":GIT_HEAD,"epoch":epoch,"plane":plane,"model_key":model_key,"output_name":output_name,"model_state_dict":model_state,"optimizer_state_dict":opt.state_dict(),"scheduler_state_dict":sch.state_dict(),"best":best,"best_state_dict":best_state,"bad":bad,"history":history,"class_weights":weights.detach().cpu().tolist(),"num_classes":num_classes,"base_channels":CFG.base_channels,"target_size":CFG.target_size,"monitor_metric":monitor_metric,"resume_kind":resume_kind}

def write_resume_state(*, plane:str, epoch:int, best:float, bad:int, history:list[dict[str,Any]], last_path:Path, best_path:Path|None)->Path:
    state={"schema_version":1,"run_id":CFG.run_id,"plane":plane,"epoch":epoch,"best":best,"bad":bad,"history_length":len(history),"last_checkpoint":last_path.name,"best_checkpoint":best_path.name if best_path else None,"updated_at_utc":utc_now()}
    dest=CFG.resume_dir/f"{plane}_resume_state.json"; atomic_write_json(state,dest); return dest

_LAST_SYNCED_EPOCH:dict[str,int]={}
def sync_resume_if_required(*, epoch:int, last_path:Path, best_path:Path|None, force:bool=False)->None:
    if not CFG.cloud_mode or not CFG.sync_resume or CFG.preflight_only: return
    if not force and epoch % CFG.sync_every_n_epochs != 0: return
    if _LAST_SYNCED_EPOCH.get(str(last_path)) == epoch and not force: return
    paths=[p for p in [last_path,best_path] if p is not None and p.exists()]
    if not paths: return
    wait_for_minimum_file_age(paths, CFG.sync_min_file_age_seconds)
    run_sync_command(repo_root=CFG.repo_root, env_file=CFG.training_env_file, mode="push-resume", execute=not CFG.sync_dry_run, failure_is_fatal=CFG.sync_failure_is_fatal)
    _LAST_SYNCED_EPOCH[str(last_path)] = epoch

def train_one_model(*,plane:str,model_key:str,records:list[SliceRecord],classes:list[str],label_map:dict[int,int]|None,output_name:str,raw0_boost:float=1.0)->dict[str,Any]:
    require_cuda_for_training()
    num_classes=len(classes); train_loader,val_loader,test_loader=loaders(records,num_classes,label_map)
    out=CFG.models_dir/output_name
    last_path,best_path=_resume_paths(output_name)
    resume_payload=None
    if CFG.resume_training and last_path.exists():
        print(f"RESUME: cargando checkpoint incremental {last_path}")
        resume_payload=torch.load(last_path,map_location=DEVICE,weights_only=False)
    if resume_payload and "class_weights" in resume_payload:
        weights=torch.tensor(resume_payload["class_weights"],dtype=torch.float32).to(DEVICE)
        print("RESUME: class_weights recuperados", [round(float(x),3) for x in weights.detach().cpu().tolist()])
    else:
        weights=estimate_weights([r for r in records if r.split=="train"],num_classes,label_map,{1:raw0_boost} if plane=="axial" and raw0_boost>1 else None).to(DEVICE)
    model=build_model(plane,num_classes,CFG.base_channels).to(DEVICE); ce=nn.CrossEntropyLoss(weight=weights)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG.lr,weight_decay=1e-4); sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=0.5,patience=4)
    best=-1.0; best_state=None; bad=0; history=[]; start_epoch=1
    if resume_payload:
        model.load_state_dict(resume_payload["model_state_dict"],strict=True)
        opt.load_state_dict(resume_payload["optimizer_state_dict"])
        sch.load_state_dict(resume_payload["scheduler_state_dict"])
        best=float(resume_payload.get("best",-1.0))
        best_state=resume_payload.get("best_state_dict")
        bad=int(resume_payload.get("bad",0))
        history=list(resume_payload.get("history",[]))
        start_epoch=int(resume_payload.get("epoch",0))+1
        print(f"RESUME: {plane} continua desde epoch {start_epoch}; best={best}; bad={bad}; history={len(history)}")
    if start_epoch>CFG.max_epochs:
        print(f"RESUME: start_epoch={start_epoch} supera max_epochs={CFG.max_epochs}; se evalua el mejor checkpoint disponible.")
    monitor_metric="dice_macro_excluding_raw0" if plane=="axial" else "dice_macro_no_bg"
    for epoch in range(start_epoch,CFG.max_epochs+1):
        model.train(); losses=[]
        total_batches=len(train_loader)
        for batch_idx,(x,y) in enumerate(train_loader, start=1):
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True); logits=model(x)
            loss=0.5*ce(logits,y)+0.5*dice_loss(logits,y,num_classes); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); losses.append(float(loss.detach().cpu()))
            if batch_idx == 1 or batch_idx % CFG.train_log_every_n_batches == 0 or batch_idx == total_batches:
                print(f"epoch {epoch} batch {batch_idx}/{total_batches} loss {float(loss.detach().cpu()):.4f}")
        vm=evaluate(model,val_loader,num_classes)
        score=vm.get(monitor_metric) or 0.0
        sch.step(score)
        history.append({"epoch":epoch,"train_loss":float(np.mean(losses)),"val":vm,"monitor_metric":monitor_metric,"monitor_score":score}); print(epoch, history[-1]["train_loss"], monitor_metric, score)
        improved=score>best
        if improved:
            best=score; best_state=_cpu_state_dict(model); bad=0
            atomic_torch_save(_checkpoint_common(epoch=epoch,plane=plane,model_key=model_key,output_name=output_name,model_state=best_state,opt=opt,sch=sch,best=best,best_state=best_state,bad=bad,history=history,weights=weights,num_classes=num_classes,monitor_metric=monitor_metric,resume_kind="best"), best_path)
            print(f"BEST: {monitor_metric}={best:.6f}; guardado {best_path}")
        else:
            bad+=1
        atomic_torch_save(_checkpoint_common(epoch=epoch,plane=plane,model_key=model_key,output_name=output_name,model_state=_cpu_state_dict(model),opt=opt,sch=sch,best=best,best_state=best_state,bad=bad,history=history,weights=weights,num_classes=num_classes,monitor_metric=monitor_metric,resume_kind="last"), last_path)
        print(f"LAST: epoch {epoch} guardado {last_path}")
        write_resume_state(plane=plane,epoch=epoch,best=best,bad=bad,history=history,last_path=last_path,best_path=best_path if best_path.exists() else None)
        sync_resume_if_required(epoch=epoch,last_path=last_path,best_path=best_path if best_path.exists() else None)
        if bad>=CFG.patience: print("early stopping"); break
    if best_state is None and best_path.exists():
        best_payload=torch.load(best_path,map_location="cpu",weights_only=False)
        best_state=best_payload.get("best_state_dict") or best_payload.get("model_state_dict")
        best=float(best_payload.get("best",best))
        history=list(best_payload.get("history",history))
    if best_state is None:
        best_state=_cpu_state_dict(model)
        print("WARNING: no habia best_state; se usa el estado actual del modelo.")
    model.load_state_dict(best_state,strict=True); model.to(DEVICE); tm=evaluate(model,test_loader,num_classes)
    ckpt={"checkpoint_schema_version":2,"run_id":CFG.run_id,"saved_at_utc":utc_now(),"git_commit":GIT_HEAD,"model_state_dict":_cpu_state_dict(model),"num_classes":num_classes,"base_channels":CFG.base_channels,"target_size":CFG.target_size,"class_weights":weights.detach().cpu().tolist(),"classes":classes,"metrics":{"best_val_monitor_score":best,"early_stopping_metric":monitor_metric,"test":tm,"history":history,"quality_gate_target_dice_macro_no_bg":0.70,"quality_gate_note":"Axial reports dice_macro_no_bg and dice_macro_excluding_raw0; gate final se define sobre split curado E9 por paciente."},"model_key":model_key,"plane":plane,"seed":CFG.seed,"created_at_unix":int(time.time()),"resume":{"enabled":CFG.resume_training,"last_checkpoint":str(last_path),"best_checkpoint":str(best_path)}}
    if plane=="sagittal":
        ckpt["label_group_mapping"]={str(k):int(v) for k,v in (label_map or {}).items()}
        ckpt["sagittal_axis"]=CFG.spider_sagittal_axis
        ckpt["slice_strategy"]="foreground_slices_after_canonicalize_spider_array"
    if plane=="axial": ckpt["label_map"]={str(k):int(v) for k,v in (label_map or {0:0,1:1,2:2,3:3,4:4,5:5}).items()}; ckpt["label_map_note"]="E9 curado esperado en 0..5; pairing crudo usa 250,0,50,100,150,200 -> 0..5."
    atomic_torch_save(ckpt,out)
    loaded=torch.load(out,map_location="cpu",weights_only=False); strict_model,meta=build_checkpoint_model(model_key,loaded); strict_model.eval()
    with torch.inference_mode(): shape=list(strict_model(torch.randn(1,1,*CFG.target_size)).shape)
    digest=sha256(out.read_bytes()).hexdigest(); result={"artifact":str(out),"sha256":digest,"metrics":tm,"strict_load_smoke_shape":shape,"runtime_meta":meta,"resume":{"last_checkpoint":str(last_path),"best_checkpoint":str(best_path)}}
    atomic_write_json(result, CFG.manifests_dir/f"{out.stem}.metrics.json"); sync_resume_if_required(epoch=int(history[-1]["epoch"]) if history else 0,last_path=last_path,best_path=best_path if best_path.exists() else None,force=True); print(json.dumps(result,indent=2)); return result


## 7. Entrenamiento sagital SPIDER


In [ ]:
RUN_SAGITTAL = env_bool("RUN_SAGITTAL", True)
if RUN_SAGITTAL:
    spider_records, spider_label_map = build_spider_records()
    summarize_records(spider_records, "SPIDER sagittal")
    freeze_records(spider_records, CFG.manifests_dir / "spider_sagittal_patient_split.csv")
    sagittal_result = train_one_model(plane="sagittal",model_key="sagittal_spider",records=spider_records,classes=SAGITTAL_CLASSES,label_map=spider_label_map,output_name="sagittal_spider_multiclass_final_best.pt")
else:
    sagittal_result = None


## 8. Entrenamiento axial ALKAFRI/Sudirman


In [ ]:
RUN_AXIAL = env_bool("RUN_AXIAL", True)
if RUN_AXIAL:
    axial_records, axial_label_map = build_axial_records()
    summarize_records(axial_records, "ALKAFRI axial")
    freeze_records(axial_records, CFG.manifests_dir / "alkafri_axial_patient_split.csv")
    axial_result = train_one_model(plane="axial",model_key="axial_t2_alkafri",records=axial_records,classes=AXIAL_CLASSES,label_map=axial_label_map,output_name="axial_t2_alkafri_final_best.pt",raw0_boost=float(os.getenv("AXIAL_RAW0_WEIGHT_BOOST", "3.0")))
else:
    axial_result = None


## 9. Quality gate, manifest y sync final


In [ ]:
def artifact_sha(path:Path)->str:
    return sha256(path.read_bytes()).hexdigest()

def quality_gate_summary(results:dict[str,dict[str,Any]])->pd.DataFrame:
    rows=[]
    for plane,result in results.items():
        m=result["metrics"]
        rows.append({"plane":plane,"artifact":Path(result["artifact"]).name,"sha256":result["sha256"],"dice_macro_no_bg":m.get("dice_macro_no_bg"),"iou_macro_no_bg":m.get("iou_macro_no_bg"),"dice_macro_excluding_raw0":m.get("dice_macro_excluding_raw0"),"passes_0_70_no_bg":(m.get("dice_macro_no_bg") or 0)>=0.70,"passes_0_70_excluding_raw0":(m.get("dice_macro_excluding_raw0") or 0)>=0.70})
    df=pd.DataFrame(rows); atomic_write_dataframe_csv(df, CFG.manifests_dir/"final_training_quality_gate_summary.csv", index=False); return df

def write_training_run_manifest(results:dict[str,dict[str,Any]], summary:pd.DataFrame)->Path:
    artifacts=[]
    for folder in [CFG.models_dir, CFG.resume_dir, CFG.manifests_dir]:
        for p in sorted(folder.glob("*")):
            if p.is_file() and not p.name.startswith("."):
                artifacts.append({"filename":p.name,"size_bytes":p.stat().st_size,"sha256":artifact_sha(p)})
    data={"schema_version":1,"run_id":CFG.run_id,"created_at_utc":utc_now(),"runtime":CFG.runtime,"git":{"commit":GIT_HEAD,"branch":GIT_BRANCH,"dirty":GIT_DIRTY},"config":{"base_channels":CFG.base_channels,"target_size":CFG.target_size,"batch_size":CFG.batch_size,"max_epochs":CFG.max_epochs,"patience":CFG.patience,"lr":CFG.lr,"seed":CFG.seed,"sync_dry_run":CFG.sync_dry_run,"sync_every_n_epochs":CFG.sync_every_n_epochs},"results":results,"quality_gate":summary.to_dict(orient="records"),"artifacts":artifacts}
    dest=CFG.manifests_dir/f"training_run_{CFG.run_id}.json"; atomic_write_json(data,dest); return dest

results = {k: v for k, v in {"sagittal": sagittal_result, "axial": axial_result}.items() if v is not None}
if results:
    summary = quality_gate_summary(results)
    manifest_path = write_training_run_manifest(results, summary)
    display(summary)
    if CFG.cloud_mode and CFG.sync_final_artifacts:
        wait_for_minimum_file_age([Path(v["artifact"]) for v in results.values()] + [manifest_path], CFG.sync_min_file_age_seconds)
        run_sync_command(repo_root=CFG.repo_root, env_file=CFG.training_env_file, mode="push-final", execute=not CFG.sync_dry_run, failure_is_fatal=CFG.sync_failure_is_fatal)
else:
    print("SKIP quality gate: no hay planos activos o entrenamiento omitido")


## Instrucciones post-entrenamiento

Los `.pt` finales quedan en `CFG.models_dir`, checkpoints incrementales en `CFG.resume_dir`, manifests en `CFG.manifests_dir` y logs futuros en `CFG.logs_dir`. No commitear `.pt`, datasets ni outputs pesados. Para corrida real en VM cambiar explicitamente `PFI_PREFLIGHT_ONLY=0` y `PFI_SYNC_DRY_RUN=0` en `training-vm.env` luego de validar preflight y dry-run.
